<div dir="rtl" style="text-align:right">

# کارگاه جست‌وجو — مقایسه BFS، DFS، UCS و A*
**جلسه ۹ درس «مقدمه‌ای بر هوش مصنوعی و کاربرد آن در مهندسی پزشکی»**

این نوت‌بوک نسخه‌ی سلول‌به‌سلول اسکریپت `session_09_search_workshop.py` است. سلول‌ها را به‌ترتیب اجرا کنید (`Shift + Enter`). فقط از کتابخانه استاندارد پایتون استفاده می‌شود و به نصب کتابخانه‌ی اضافه نیاز نیست.

</div>

In [ ]:
import sys
import heapq
import itertools
import time
from collections import deque

print("کتابخانه‌های استاندارد آماده‌اند. Python:", sys.version.split()[0])

<div dir="rtl" style="text-align:right">

## ۱. مسئله: مسیریابی در یک بیمارستان
می‌خواهیم از **ورودی اصلی (S)** به **آزمایشگاه (G)** برویم. وزن هر یال فاصله‌ی تقریبی راهرو بر حسب متر است و یال‌ها دوبطرفه‌اند. بخش‌های **انبار (H)** و **نمازخانه (I)** بن‌بست‌اند و برای رسیدن به آزمایشگاه بی‌فایده‌اند — در ادامه می‌بینیم چنین گره‌هایی هزینه‌ی جست‌وجوی UCS را بالا می‌برند.

</div>

In [ ]:
# گراف بیمارستان: گره‌ها بخش‌ها و وزن یال‌ها فاصله (متر) است
GRAPH = {
    "S": {"A": 2, "B": 5},
    "A": {"S": 2, "C": 4, "D": 7, "H": 4},
    "B": {"S": 5, "D": 3, "E": 6, "I": 2},
    "C": {"A": 4, "F": 3},
    "D": {"A": 7, "B": 3, "F": 2, "G": 4},
    "E": {"B": 6, "G": 7},
    "F": {"C": 3, "D": 2, "G": 7},
    "G": {"D": 4, "E": 7, "F": 7},
    "H": {"A": 4},
    "I": {"B": 2},
}
START, GOAL = "S", "G"

# برآورد هیوریستیک فاصله تا G برای A* (متر) — مجاز و سازگار
HEURISTIC = {
    "S": 9, "A": 7, "B": 6, "C": 7, "D": 4,
    "E": 7, "F": 5, "G": 0, "H": 11, "I": 8,
}
print("گراف با", len(GRAPH), "گره تعریف شد.")

<div dir="rtl" style="text-align:right">

## ۲. جست‌وجوی اول-سطح و اول-عمق
هر دو الگوریتم وزن یال‌ها را نمی‌بینند؛ فقط ترتیب گشتن متفاوت است:
- **BFS** با **صف**: نزدیک‌ترین گره‌ها (از نظر تعداد یال) اول بررسی می‌شوند.
- **DFS** با **پشته**: تا عمق ممکن پیش می‌رود.

هر دو تعداد «گره‌های بسط‌یافته» را برمی‌گردانند تا بعداً با UCS و A* مقایسه کنیم.

</div>

In [ ]:
def reconstruct(came_from, node):
    """بازسازی مسیر از دیکشنری والد‌ها."""
    path = []
    while node is not None:
        path.append(node)
        node = came_from[node]
    return path[::-1]


def path_cost(path, graph):
    """هزینه کل مسیر از روی وزن یال‌ها."""
    return sum(graph[a][b] for a, b in zip(path, path[1:]))


def bfs(graph, start, goal):
    """جست‌وجوی اول-سطح با صف؛ کم‌یال‌ترین مسیر را می‌دهد."""
    frontier = deque([start])
    came_from = {start: None}
    expanded = 0
    while frontier:
        node = frontier.popleft()
        if node == goal:
            return reconstruct(came_from, node), expanded
        expanded += 1
        for nxt in sorted(graph[node]):
            if nxt not in came_from:
                came_from[nxt] = node
                frontier.append(nxt)
    return None, expanded


def dfs(graph, start, goal):
    """جست‌وجوی اول-عمق با پشته؛ مسیر تضمیناً بهینه نیست."""
    frontier = [start]
    came_from = {start: None}
    expanded = 0
    while frontier:
        node = frontier.pop()
        if node == goal:
            return reconstruct(came_from, node), expanded
        expanded += 1
        # معکوس push می‌کنیم تا همسایه الفبایی کوچک‌تر زودتر pop شود
        for nxt in sorted(graph[node], reverse=True):
            if nxt not in came_from:
                came_from[nxt] = node
                frontier.append(nxt)
    return None, expanded

In [ ]:
for name, fn in [("BFS", bfs), ("DFS", dfs)]:
    path, expanded = fn(GRAPH, START, GOAL)
    print(name, "->", " -> ".join(path), "| cost:", path_cost(path, GRAPH),
          "| expanded:", expanded)

<div dir="rtl" style="text-align:right">

## ۳. جست‌وجوی یکنواخت هزینه و A*
این دو از **صف اولویت** استفاده می‌کنند و وزن یال‌ها را می‌بینند:
- **UCS** بر اساس $g$ (هزینه‌ی واقعی از مبدأ) مرتب می‌کند و مسیر بهینه را تضمین می‌کند.
- **A*** بر اساس $g + h$ مرتب می‌کند؛ اگر $h$ هرگز بیشتر از فاصله‌ی واقعی تا هدف نباشد (هیوریستیک مجاز)، بهینگی حفظ می‌شود و جست‌وجو هوشمندانه‌تر می‌شود.

</div>

In [ ]:
def ucs(graph, start, goal):
    """جست‌وجوی یکنواخت هزینه با صف اولویت؛ بهینه."""
    counter = itertools.count()  # برای شکستن تساوی‌ها به‌صورت قطعی
    frontier = [(0, next(counter), start)]
    best_g = {start: 0}
    came_from = {start: None}
    expanded = 0
    while frontier:
        g, _, node = heapq.heappop(frontier)
        if g > best_g.get(node, float("inf")):  # ورودی کهنه؛ نادیده بگیر
            continue
        if node == goal:
            return reconstruct(came_from, node), expanded
        expanded += 1
        for nxt, w in sorted(graph[node].items()):
            new_g = g + w
            if new_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = new_g
                came_from[nxt] = node
                heapq.heappush(frontier, (new_g, next(counter), nxt))
    return None, expanded


def a_star(graph, start, goal, heuristic):
    """A*: UCS + هیوریستیک؛ با h مجاز، مسیر بهینه با بسط کمتر."""
    counter = itertools.count()
    frontier = [(heuristic[start], next(counter), start)]
    best_g = {start: 0}
    came_from = {start: None}
    expanded = 0
    while frontier:
        f, _, node = heapq.heappop(frontier)
        g = best_g.get(node, float("inf"))
        if f - heuristic[node] > g:  # ورودی کهنه؛ نادیده بگیر
            continue
        if node == goal:
            return reconstruct(came_from, node), expanded
        expanded += 1
        for nxt, w in sorted(graph[node].items()):
            new_g = g + w
            if new_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = new_g
                came_from[nxt] = node
                heapq.heappush(frontier, (new_g + heuristic[nxt], next(counter), nxt))
    return None, expanded

In [ ]:
for name, fn in [("UCS", lambda: ucs(GRAPH, START, GOAL)),
                 ("A*", lambda: a_star(GRAPH, START, GOAL, HEURISTIC))]:
    path, expanded = fn()
    print(name, "->", " -> ".join(path), "| cost:", path_cost(path, GRAPH),
          "| expanded:", expanded)

<div dir="rtl" style="text-align:right">

## ۴. جدول مقایسه
حالا هر چهار الگوریتم را روی همان مسئله اجرا می‌کنیم و مسیر، هزینه، تعداد گره‌های بسط‌یافته و زمان (با `time.perf_counter`) را کنار هم می‌گذاریم. انتظار شما چیست؟ اجرا کنید و با پیش‌بینی‌تان مقایسه کنید.

</div>

In [ ]:
algorithms = [
    ("BFS", lambda: bfs(GRAPH, START, GOAL)),
    ("DFS", lambda: dfs(GRAPH, START, GOAL)),
    ("UCS", lambda: ucs(GRAPH, START, GOAL)),
    ("A*", lambda: a_star(GRAPH, START, GOAL, HEURISTIC)),
]

header = "{:<10}{:<22}{:>6}{:>6}{:>10}{:>12}".format(
    "Algorithm", "Path", "Edges", "Cost", "Expanded", "Time(ms)")
print(header)
print("-" * len(header))
for name, run in algorithms:
    t0 = time.perf_counter()
    path, expanded = run()
    elapsed_ms = (time.perf_counter() - t0) * 1000
    print("{:<10}{:<22}{:>6}{:>6}{:>10}{:>12.3f}".format(
        name, " -> ".join(path), len(path) - 1, path_cost(path, GRAPH),
        expanded, elapsed_ms))

<div dir="rtl" style="text-align:right">

## ۵. آزمایش: هیوریستیک صفر چه می‌کند؟
اگر همه‌ی $h$ها را صفر کنیم، ترتیب صف اولویت فقط بر اساس $g$ است؛ یعنی A* باید دقیقاً مثل UCS رفتار کند. اجرا کنید و تعداد گره‌های بسط‌یافته را با جدول بالا مقایسه کنید.

**آزمایش‌های بیشتر:** ۱) وزن یال S–B را از ۵ به ۲ تغییر دهید و دوباره اجرا کنید. ۲) یک یال مستقیم بین H و G با وزن ۱ اضافه کنید؛ UCS چه مسیری پیدا می‌کند و آیا `HEURISTIC` فعلی هنوز مجاز است؟

</div>

In [ ]:
h_zero = {node: 0 for node in HEURISTIC}
for name, h in [("A* با h=0 (معادل UCS)", h_zero), ("A* با هیوریستیک واقعی", HEURISTIC)]:
    path, expanded = a_star(GRAPH, START, GOAL, h)
    print(name, "->", " -> ".join(path), "| cost:", path_cost(path, GRAPH),
          "| expanded:", expanded)

<div dir="rtl" style="text-align:right">

## جمع‌بندی
- BFS مسیر کم‌یال می‌دهد اما هزینه بهینه نیست؛ DFS سریع است ولی هیچ تضمینی ندارد.
- UCS مسیر بهینه را تضمین می‌کند اما تقریباً همه گره‌ها را بسط می‌دهد.
- A* با هیوریستیک مجاز همان مسیر بهینه را با گره‌های بسط‌یافته‌ی بسیار کمتر پیدا می‌کند.
- خروجی ستون زمان به سخت‌افزار وابسته است؛ ستون مهم برای مقایسه «Expanded» است.

جزئیات بیشتر، تفسیر نتیجه‌ها و تکلیف در فایل جلسه `session_09_search_workshop.md` آمده است.

</div>